# 用 PyGMT 畫自己的地圖

台灣的地震都發生在哪裡？山脈往海底延伸，又會是什麼模樣？這堂課，我們用幾行 Python，把資料變成看得見的地圖。

從一個空白外框開始，親手加上海岸線、地震與地形，再換個角度看立體的台灣。先試著自己改指令，看懂每一步如何改變畫面；最後再邀請 AI 幫忙，把地圖帶到你想探索的世界角落。

不必一開始就看懂所有程式。先跑出一張圖、改一個地方，看看會發生什麼。

### 開始動手

- 在 Colab 先另存自己的副本，再由上往下執行。
- 第 1–4 主題先親手操作、不用 AI；第 5 主題再使用 Codex 或 Colab Gemini 延伸自己的想法。

想先看看這些工具能畫出什麼？從 [GMT、PyGMT 與地圖作品導覽](https://github.com/jimmy60504/pygmt-map-lab/blob/main/intro.md) 開始，再回來準備環境、畫出第一張圖。

### 常用快捷鍵

| 操作 | Windows／Linux | Mac |
| --- | --- | --- |
| 執行目前儲存格並移到下一格 | Shift + Enter | Shift + Enter |
| 取消／切換註解 | Ctrl + / | ⌘ + / |

把游標放在程式行，或選取多行，再按切換註解的快捷鍵，即可移除或加上行首的 `#`。只選程式行，不要連中文說明一起取消註解。

修改後按 **Shift + Enter** 看結果，等執行完成再繼續下一步。


## 0. 準備環境（直接執行，不必逐行看懂）
下面兩格負責安裝套件，課堂先不講解；完成後從第 1 節開始看程式。

**Colab 請分開執行。** 第一格安裝 Conda 後可能自動重啟；等重新連線，再執行第二格。勿在安裝期間重複按執行。

本機已裝好環境時會跳過安裝。此教材使用 PyGMT 0.17 / GMT 6.5；Colab 安裝流程仍需於課前用實際學生帳號確認。


In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    subprocess.check_call([
        "mamba", "install", "-y", "-c", "conda-forge",
        "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "pandas", "pillow"
    ])
else:
    print("跳過 Colab 安裝。")

import pygmt  # 安裝完成後載入繪圖套件


## 1. 從空白底圖，一步一步畫出台灣
先執行下一格，看到最簡單的地圖外框。接著依序移除步驟 1–5 程式行開頭的 `#`，每次只開啟一步，保留前面已開啟的步驟，再重跑整格觀察差異。

- `fig = pygmt.Figure()`：建立一張新圖；每次重跑都從頭畫，不會累積上次的內容。
- `region`：繪圖範圍，順序是 **西、東、南、北**；`projection="M15c"`：麥卡托投影、圖寬 15 公分。
- `fig.basemap()` 畫框線、刻度等；`fig.coast()` 畫海陸與海岸線；最後 `fig.show()` 顯示結果。

**先記住繪圖順序**：在同一個 `fig` 上，後畫的內容可能蓋住前面的內容。因此先填海陸顏色，再加線條、格線與標題。

後面的呼叫省略 `region` 與 `projection`，是沿用這張圖已設定的範圍與投影，**不是讀取上一層的圖片**。圖層疊加與設定沿用是兩件事。

### 第一張圖的 API 參考
不用整頁讀完：想改哪個效果，就點對應指令，在 **Parameters** 找參數，再看 **Examples**。以下連結對應課堂使用的 PyGMT 0.17。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [pygmt.Figure()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.html) | 建立一張圖 | Methods：還能加哪些內容 |
| [fig.basemap()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.basemap.html) | 底圖、刻度、格線與標題 | `region`、`projection`、`frame` |
| [fig.coast()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.coast.html) | 海陸填色與海岸線 | `land`、`water`、`shorelines`、`resolution` |
| [fig.show()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.show.html) | 顯示目前的圖 | `width`、`dpi` |
| [fig.savefig()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.savefig.html) | 存成圖片 | `fname`、`dpi` |

`import pygmt` 是載入套件，不是繪圖指令。`region=[119, 123, 21, 26]` 依序指定西界、東界、南界、北界。

**試著發現一個新選項**：打開 `coast` 文件，找找 `borders` 或 `rivers` 能做什麼。


In [ ]:
import pygmt

fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame="0")

# 1. 加上經緯度數字與刻度（a：標示數字；f：刻度）
# fig.basemap(frame="af")

# 2. 填上陸地與海洋顏色
# fig.coast(land="gray90", water="lightblue", resolution="h")

# 3. 加上海岸線
# fig.coast(shorelines="0.6p,gray30", resolution="h")

# 4. 加上每 1 度的經緯格線，並重畫刻度（g：格線）
# fig.basemap(frame="a1f0.5g1")

# 5. 加上標題
# fig.basemap(frame="+tTaiwan: coastlines")

fig.show()
# 圖片顯示後，可按右鍵另存圖片。


## 2. 地震在哪裡？

### 先挑一個你想畫的地震事件

不知道要選哪個事件？台灣事件可以先到氣象署看看：

- [最近地震](https://www.cwa.gov.tw/V8/C/E/index.html)：從近期有感地震找題目，點進報告看發生時間、位置與規模。
- [歷史災害地震](https://scweb.cwa.gov.tw/zh-tw/page/disaster)：認識集集、美濃等重大事件，找一個你想進一步觀察的地震。較早期的歷史記載不一定有完整的 USGS 資料，練習可先選近代事件。

想畫世界其他地方，則可以從 USGS 找題目：

- [Latest Earthquakes｜最近地震地圖](https://earthquake.usgs.gov/earthquakes/map/)：瀏覽近期全球地震，可選「30 Days, Significant Worldwide」找最近一個月的重要事件。
- [Significant Earthquakes｜重大地震](https://earthquake.usgs.gov/earthquakes/browse/significant.php)：依年份找事件。「重大」不只看規模，也考慮有感回報與可能影響。
- [Search Earthquake Catalog｜地震目錄查詢](https://earthquake.usgs.gov/earthquakes/search/)：設定時間、規模與區域，也可以選 CSV 輸出。

選好後，記下時間與震央位置，把下方網址改成事件前後幾天、震央附近的範圍，觀察主震周邊的地震分布；地圖的 `region` 也要配合調整。

**時間要對齊**：氣象署報告使用台灣時間（UTC+8），下方 USGS 查詢使用 UTC，要先減 8 小時，日期也可能變成前一天。這裡用氣象署找題目，實際繪圖資料仍來自 USGS；兩邊的規模與位置可能不同，不必強求完全一致。


### 用 USGS API 取得資料

向 USGS 要一份地震資料，再把經緯度畫到地圖上。API 就像點餐：在網址指定時間、區域與最低規模，USGS 就回傳符合條件的 CSV 表格。

這次查詢 **2020-01-01 至執行當下（UTC）、規模 ≥ 2**，範圍是東經 119–123 度、北緯 21–26 度，與前面的地圖相同。觀察地震位置與深度的空間變化；USGS 對台灣小地震的收錄不完整，這些點不代表所有台灣地震，也不是完整的隱沒板塊形狀。這組圖呈現多年地震分布；第一張另外標記 2024 花蓮主震來示範符號，不代表其他地震都是它的餘震。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 按條件取得地震 CSV | `format`、`starttime`、`endtime`、`minmagnitude`、經緯度範圍 |
| [pd.read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) | 把 CSV 網址讀成表格 | `filepath_or_buffer`：此處傳入 `url` |

可以複製下格印出的網址到瀏覽器下載資料；每次執行都需要網路。


In [ ]:
import pandas as pd
from datetime import datetime, timezone


# 1. 設定結束時間：每次執行當下的 UTC 時間
endtime = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")


# 2. 組合查詢網址：修改下方時間、規模與範圍
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query"  # USGS 地震查詢入口
    "?format=csv"                    # 回傳 CSV 表格；第一個參數用 ?
    "&starttime=2020-01-01"           # 開始時間（UTC）；後續參數用 &
    f"&endtime={endtime}"             # 結束時間（UTC）
    "&minmagnitude=2"                 # 最低地震規模
    "&minlongitude=119"               # 西界：最小經度
    "&maxlongitude=123"               # 東界：最大經度
    "&minlatitude=21"                 # 南界：最小緯度
    "&maxlatitude=26"                 # 北界：最大緯度
    "&orderby=time-asc"              # 依時間由早到晚排列
)


# 3. 讀取資料，去掉缺少繪圖欄位的地震
quakes = pd.read_csv(url)
quakes = quakes.dropna(subset=["longitude", "latitude", "mag", "depth"])


# 4. 確認查詢網址、筆數與前五筆資料
print(url)  # 可複製到瀏覽器，查看同一份 CSV
print(f"可繪製的地震：{len(quakes)} 筆；深度單位：km；時間：UTC。")

quakes[["time", "longitude", "latitude", "mag", "depth"]].head()


### 第一張：從 Pandas 表格取經緯度畫點
`quakes` 是 Pandas 的 DataFrame：每列是一筆地震，欄位包含經度、緯度、規模與深度。

這裡不是用 Pandas 畫圖，而是把 `quakes.longitude` 與 `quakes.latitude` 交給 PyGMT 的 `fig.plot()`。先讓所有點一樣大、同一種顏色，只看地震在哪裡。

**圓圈之外，也能畫星星**：`style="c0.12c"` 是直徑 0.12 cm 的圓圈，`style="a0.6c"` 是大小 0.6 cm 的星形。下格用星星標出 [2024 花蓮主震（USGS）](https://earthquake.usgs.gov/earthquakes/eventpage/us7000m9g4/executive)，示範直接指定經緯度。星星最後畫，會疊在圓圈上。

**試看看**：將星星的 `a` 改成 `t`（三角形）或 `s`（正方形），保持 `0.6c` 不變。每張圖都重新建立 `fig`，不會疊到上一張；第二、三張先不加這個標記。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 畫點與不同符號 | `x`、`y`、`style`、`fill`、`pen`、`label` |
| [fig.legend()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.legend.html) | 顯示符號圖例 | `position`、`box` |


In [ ]:
import pygmt

fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")


# 從 Pandas 表格取出經緯度，每一列畫成一個點
fig.plot(
    x=quakes.longitude,  # 經度欄位：每筆地震的 x 位置
    y=quakes.latitude,  # 緯度欄位：每筆地震的 y 位置
    style="c0.12c",  # 所有圓圈的直徑都一樣：0.12 cm
    fill="tomato",  # 固定顏色：所有地震都用同一種色
    pen="0.25p,gray20",
    transparency=40,
)


# 單獨標出 2024 花蓮主震（USGS：us7000m9g4）
fig.plot(
    x=121.5976,       # 經度
    y=23.8356,        # 緯度
    style="a0.6c",    # a：星形；t：三角形；s：正方形
    fill="yellow",
    pen="1p,black",
    label="2024 Hualien mainshock",  # 新增：把星星與文字放進圖例
)

fig.legend(position="JTL+jTL+o0.2c", box="+gwhite+p0.5p")


fig.show()
fig.savefig("02a_earthquake_points.png", dpi=150)


### 第二張：讓地震規模決定大小
把固定大小的 `style="c0.12c"` 改成 `style="c"`，另外用 `size` 傳入每筆地震的圓圈直徑；顏色先維持不變。

用 `if / elif / else` 分區間，同一區間用固定直徑：M<3 → 0.035 cm、3≤M<4 → 0.07 cm、4≤M<5 → 0.14 cm、5≤M<6 → 0.28 cm、6≤M<7 → 0.56 cm、M≥7 → 1.12 cm。`plot_quakes.mag.apply(magnitude_size)` 會把每筆規模交給這個函式判斷。這是視覺設計，不是能量比例或影響半徑；規模也不是各地感受到的震度。

**試看看**：`ascending=True` 讓小圓先畫、大圓後畫；改成 `False`，比較重疊效果。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 讓每筆地震有不同大小 | `size`、`style`、`transparency` |
| [DataFrame.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) | 決定資料與繪圖順序 | `by`：此處為 `"mag"`；`ascending` |


In [ ]:
import pygmt

# 依規模區間指定圓圈直徑（cm）；同一區間使用相同大小
def magnitude_size(magnitude):
    if magnitude < 3:
        return 0.035
    elif magnitude < 4:
        return 0.07
    elif magnitude < 5:
        return 0.14
    elif magnitude < 6:
        return 0.28
    elif magnitude < 7:
        return 0.56
    else:
        return 1.12


# 小圓先畫，大圓後畫
plot_quakes = quakes.sort_values("mag", ascending=True)


fig = pygmt.Figure()
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")


# 改用 size，讓每筆地震有自己的圓圈大小
fig.plot(
    x=plot_quakes.longitude,
    y=plot_quakes.latitude,
    style="c",  # 改動：只指定圓形，大小改由下一行決定
    size=plot_quakes.mag.apply(magnitude_size),  # 新增：把每筆規模換成圓圈直徑
    fill="tomato",  # 沿用：這張先不改顏色
    pen="0.25p,gray20",
    transparency=40,
)


fig.show()
fig.savefig("02b_earthquake_sizes.png", dpi=150)


### 第三張：用深度上色，認識 CPT

前一張用規模控制大小；這張保留大小，再用 `fill=plot_quakes.depth` 與 `cmap=True` 把深度轉成顏色，最後加上 color bar。

CPT（Color Palette Table）決定數值對應什麼顏色。從下方色票總覽挑一組喜歡的配色，把名稱填進 `pygmt.makecpt(cmap=...)`。

- 這次用 `cmap="gmt/seis"`：隨深度增加，由紅、橙、黃、綠轉為藍，也就是淺層紅、深層藍。
- `gmt/` 是色票分類，`seis` 是名稱；也可以試 `jet` 或 `rainbow`，比較同一份資料的呈現。
- `series=[0, 150, 1]` 將色票固定在 0–150 km；`background=True` 讓超過 150 km 的地震沿用最深端顏色，不會刪除這些事件。`reverse=True` 可以反轉顏色順序。

**試看看**：只改 `cmap`，重跑下格，比較哪些深度比較醒目。深度要對照 color bar，不能只憑明暗判斷；目前圓圈有 40% 透明度，顏色也會受底圖和重疊影響。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [GMT 色票總覽](https://docs.generic-mapping-tools.org/6.5/reference/cpts.html) | 比較內建 CPT（色票圖鑑，非函式） | 色票名稱，如 `gmt/seis`、`jet` |
| [pygmt.makecpt()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.makecpt.html) | 建立數值到顏色的對應 | `cmap`、`series`、`reverse` |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 依地震深度上色 | `fill`、`cmap=True` |
| [fig.colorbar()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.colorbar.html) | 顯示色條與單位 | `frame` |


In [ ]:
import pygmt

from io import StringIO


# 沿用上一格的 magnitude_size 與 plot_quakes

# 1. 建立新圖與深度色票
# 深度色票固定為 0–150 km，方便不同圖之間比較

fig = pygmt.Figure()

# gmt/seis：淺層紅，經橙、黃、綠，轉為深層藍
pygmt.makecpt(
    cmap="gmt/seis",  # 新增：選擇色票名稱
    series=[0, 150, 1],  # 深度起點、終點、間隔（km）；上限固定 150
    background=True,  # 超過 150 km 沿用色票最深端的顏色
    continuous=True,  # 新增：建立連續漸層
)


# 2. 畫底圖與海岸線
fig.basemap(region=[119, 123, 21, 26], projection="M15c", frame=["af", f"+tTaiwan 2020 to {endtime[:10]} | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")


# 用深度控制填色，規模繼續控制大小
fig.plot(
    x=plot_quakes.longitude,
    y=plot_quakes.latitude,
    size=plot_quakes.mag.apply(magnitude_size),  # 沿用：規模仍控制大小
    style="c",
    fill=plot_quakes.depth,  # 改動：不再填固定色名，改傳入每筆深度
    cmap=True,  # 新增：用剛建立的 CPT 把深度數值轉成顏色
    pen="0.25p,gray20",
    transparency=40,  # 0 不透明，100 完全透明
)


# 3. 加上規模圖例（留出大圓需要的間距）
legend = StringIO(
    "".join(
        f"S 0.6c c {magnitude_size(mag):.3f}c gray70 0.25p,gray20 1.4c {label}\n"
        f"G {max(0.15, magnitude_size(mag) - 0.25):.2f}c\n"
        for mag, label in [
            (2, "M < 3"),
            (3, "3 <= M < 4"),
            (4, "4 <= M < 5"),
            (5, "5 <= M < 6"),
            (6, "6 <= M < 7"),
            (7, "M >= 7"),
        ]
    )
)

fig.legend(
    spec=legend,
    position="JTL+jTL+o0.2c",
    box="+gwhite+p0.5p",
)


# 4. 加上深度色條
fig.colorbar(
    frame=["xaf", "y+lDepth (km)"],  # 新增：顯示色票對應的刻度與深度單位
)


# 5. 顯示與存圖
fig.show()
fig.savefig("02_taiwan_earthquakes.png", dpi=150)


## 3. 彩色地形：高度變成顏色
`load_earth_relief()` 取得地形網格，`grdimage()` 把高度畫成顏色。`02m` 指 2 角分，**不是 2 公尺**。第一次執行需要下載 GMT 地形資料。

**試看看**：把 `shading=True` 改成 `False`，比較起伏的可讀性。深色也可能來自陰影，不能只用明暗判斷高低。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [load_earth_relief()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.datasets.load_earth_relief.html) | 載入地形與海底高程網格 | `resolution`、`region`、`registration` |
| [fig.grdimage()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdimage.html) | 將網格畫成彩色地形 | `grid`、`cmap`、`shading` |
| [fig.colorbar()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.colorbar.html) | 顯示高程色條與單位 | `frame` |


In [ ]:
import pygmt

# 1. 取得地形網格
grid = pygmt.datasets.load_earth_relief(
    resolution="02m",  # 新增：網格間距 2 角分，不是 2 公尺
    region=[119, 123, 21, 26],  # 下載範圍：西、東、南、北
    registration="gridline",  # 網格值位於格線交點，這裡先沿用
)

print("地形網格：", grid.shape, "；高程單位：m")


# 2. 用高程上色，不再逐筆畫地震點
fig = pygmt.Figure()

fig.grdimage(
    grid=grid,  # 新增：輸入整片地形網格
    region=[119, 123, 21, 26],
    projection="M15c",
    cmap="geo",  # 改動：使用海底與陸地的地形色票
    shading=True,  # 新增：加入陰影，凸顯起伏
    frame=["af", "+tTaiwan: land and seafloor"],
)


# 3. 疊上海岸線與高程色條
fig.coast(shorelines="0.5p,gray25", resolution="h")
fig.colorbar(frame=["xaf", "y+lElevation (m)"])  # 改動：現在表示高程，單位 m


fig.show()
fig.savefig("03_taiwan_relief.png", dpi=150)


## 4. 3D 地形：換個角度看台灣
用 `grdview()` 將同一份地形畫成斜視圖。`perspective=[方位角, 仰角]` 控制觀看方向，`zsize` 控制垂直尺寸。

圖中垂直方向為了辨認起伏而誇大，不能當成真實坡度。這是固定視角圖片，不是滑鼠可拖曳的模型。

**試看看**：把方位角 135 改為 225，仰角維持 35，觀察哪些山被遮住。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.grdview()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdview.html) | 繪製斜視地形表面 | `grid`、`region`、`perspective`、`zsize`、`surftype` |


In [ ]:
import pygmt

# 1. 設定觀看方向
azimuth = 135  # 方位角：繞地形從哪個方向看（度）
elevation = 35  # 仰角：觀看角度的高低（度）


# 2. 沿用上一格的 grid，改畫 3D 地形
fig = pygmt.Figure()

fig.grdview(
    grid=grid,  # 沿用：同一份地形資料
    region=[119, 123, 21, 26, -8000, 4000],  # 新增：最後兩個值是高程下、上限（m）
    projection="M15c",
    perspective=[azimuth, elevation],  # 新增：方位角、仰角
    zsize="3c",  # 新增：垂直軸畫成多高；不是實際山高
    surftype="s",  # 新增：繪製表面
    cmap="geo",  # 沿用：高程色票
    frame=["xaf", "yaf", "zaf+lElevation (m)", "+tTaiwan: 3D relief"],
)


fig.show()
fig.savefig("04_taiwan_3d.png", dpi=150)


### 選做：旋轉動畫
這格會畫 12 個方向並合成 GIF，耗時比單張圖長。可先跳過，等核心練習完成再回來。圖面會隨視角裁切，合成時置中並統一畫布大小。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [fig.grdview()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdview.html) | 重複畫不同視角 | `perspective`：每次換方位角 |
| [fig.savefig()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.savefig.html) | 儲存各角度的影格 | `fname`、`dpi`、`resize` |


In [ ]:
import pygmt

RUN_ANIMATION = False  # 想試旋轉時改成 True

if RUN_ANIMATION:
    from PIL import Image as PILImage
    from IPython.display import display, Image
    from pathlib import Path
    OUT = Path("outputs")
    OUT.mkdir(exist_ok=True)

    # 每個角度畫一張圖
    frames = []
    for angle in range(0, 360, 30):
        frame_fig = pygmt.Figure()
        frame_fig.grdview(
            grid=grid,
            region=[119, 123, 21, 26, -8000, 4000],
            projection="M12c",
            perspective=[angle, 35],  # 改動：方位角隨迴圈改變
            zsize="2.4c",
            surftype="s",
            cmap="geo",
            frame=["xaf", "yaf", "zaf", "+tTaiwan relief"],
        )

        path = OUT / f"rotation_{angle:03d}.png"
        frame_fig.savefig(str(path), dpi=100, resize="+m0.3c")
        with PILImage.open(path) as im:
            frames.append(im.convert("RGB"))

    # 統一影格尺寸，合成 GIF
    width = max(im.width for im in frames)
    height = max(im.height for im in frames)
    canvases = []
    for im in frames:
        canvas = PILImage.new("RGB", (width, height), "white")
        canvas.paste(im, ((width-im.width)//2, (height-im.height)//2))
        canvases.append(canvas)
    gif_path = OUT / "taiwan_rotation.gif"
    canvases[0].save(gif_path, save_all=True, append_images=canvases[1:],
                     duration=250, loop=0)
    display(Image(filename=str(gif_path)))
else:
    print("動畫先跳過；把 RUN_ANIMATION 改成 True 即可執行。")

## 5. 從台灣到世界：開始用 AI
先執行以下全球地形範例，再用 Codex 或 Colab Gemini 協助修改。全球採 `01d`（1 度）資料，避免一開始下載太大的網格。

**練習**：選一個台灣以外的區域，請 AI 依原程式修改範圍、投影與解析度，保留資料來源和單位。

提示詞：
> 我在 Colab 使用 PyGMT 0.17，這段程式可以執行。請以它為基礎，改畫日本周邊的彩色地形，選合適的區域投影與資料解析度，加上標題與高程色階。不要加入 ObsPy。先解釋要改哪些參數，再給程式。

請核對：畫的是指定區域嗎？色階單位正確嗎？資料來源仍然存在嗎？能指出 AI 改了哪個參數嗎？

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [load_earth_relief()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.datasets.load_earth_relief.html) | 取得全球地形 | `resolution="01d"` |
| [fig.grdimage()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdimage.html) | 把同一種繪圖方法延伸到全球 | `region`、`projection`、`grid` |


In [ ]:
import pygmt

# 1. 取得較粗的全球地形，減少資料量
world_grid = pygmt.datasets.load_earth_relief(
    resolution="01d",  # 改動：全球先用 1 度間距
    registration="gridline",
)


# 2. 改用全球範圍與投影
fig = pygmt.Figure()

fig.grdimage(
    grid=world_grid,  # 改動：使用全球網格
    region="g",  # 改動：全球範圍
    projection="W15c",  # 改動：Mollweide 全球投影，寬 15 cm
    cmap="geo",
    shading=True,
    frame=["af", "+tWorld relief"],
)


fig.coast(shorelines="0.25p,gray30", resolution="c")
fig.colorbar(frame=["xaf", "y+lElevation (m)"])

fig.show()
fig.savefig("05_world_relief.png", dpi=150)


## 6. 隔週繳交
選擇全球或自選區域，將 Notebook、成果圖與簡短說明放到 GitHub，交 repository 網址。說明繪圖區域、資料來源、主要觀察，以及 AI 協助了什麼。

- Notebook：儲存目前 `.ipynb`，保留有用的執行輸出。
- 成果圖：在左側檔案區下載各張 PNG；選做動畫在 `outputs/`。
- 地震資料：由上方查詢網址下載 CSV，並在作業說明保留該網址。
- GitHub：建立 repository，上傳 Notebook、成果與 README，確認教師可讀取連結。

下格打包圖片，方便下載；Notebook 請另外由 Colab 的檔案選單下載。

In [ ]:
import shutil
from pathlib import Path

bundle = Path("submission_assets")
bundle.mkdir(exist_ok=True)
for filename in ["01_taiwan_coast.png", "02a_earthquake_points.png",
                 "02b_earthquake_sizes.png", "02_taiwan_earthquakes.png",
                 "03_taiwan_relief.png", "04_taiwan_3d.png", "05_world_relief.png"]:
    if Path(filename).exists():
        shutil.copy2(filename, bundle / filename)
for folder in [Path("outputs")]:
    if folder.exists():
        shutil.copytree(folder, bundle / folder.name, dirs_exist_ok=True)
archive = shutil.make_archive("submission_assets", "zip", root_dir=bundle)
print("已建立：", archive)


## 資料來源與版本

- [原始課程參考 Notebook](https://github.com/oceanicdayi/plot_plate_boundary_pygmt/blob/main/pygmt_plot_plate_boundary.ipynb)
- [GMT 全球地形資料](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)：PyGMT 載入，首次使用需連網。
- [PyGMT 0.17 安裝文件](https://www.pygmt.org/v0.17.0/install.html)


| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 本課使用的真實事件資料 | 查詢條件包含在下載網址中 |

本機執行驗證與 Colab 雲端安裝驗證是兩件事；實際測試結果請參閱同資料夾 README。